# MGMT298D: Science and Strategy of AI## Week 2: XGBoost & Hyperparameter Tuning### UCLA Anderson School of Management

## 1. Imports

In [ ]:
import pandas as pdimport numpy as npfrom sklearn.preprocessing import LabelEncoderfrom sklearn.model_selection import cross_val_score, train_test_splitfrom sklearn.metrics import mean_absolute_errorfrom xgboost import XGBRegressorimport matplotlib.pyplot as pltimport seaborn as sns# Set visualization stylesns.set_style('whitegrid')%matplotlib inline

## 2. Load Data

In [ ]:
# Fetch Range Rover dataseturl = 'https://raw.githubusercontent.com/ucla-anderson-SSAI/SSAI/main/range_rover.csv'df = pd.read_csv(url)# Display dataset overviewprint(f"Dataset shape: {df.shape}")print(f"\nFirst 5 rows:")print(df.head())print(f"\nColumn names and types:")print(df.dtypes)

## 3. Encode Features

In [ ]:
# Identify target column (try 'sellingprice', fallback to 'price')target_col = 'sellingprice' if 'sellingprice' in df.columns else 'price'print(f"Target column: {target_col}")# Label-encode categorical featuresle_trim = LabelEncoder()le_state = LabelEncoder()le_color = LabelEncoder()df['trim_enc'] = le_trim.fit_transform(df['trim'])df['state_enc'] = le_state.fit_transform(df['state'])df['color_enc'] = le_color.fit_transform(df['color'])# Define features and targetfeature_cols = ['year', 'mileage', 'trim_enc', 'state_enc', 'color_enc']X = df[feature_cols].copy()y = df[target_col].copy()print(f"\nFeatures: {feature_cols}")print(f"Target: {target_col}")print(f"\nTarget statistics:")print(y.describe())

## 4. Baseline Model

In [ ]:
# Train baseline XGBRegressor with default hyperparametersxgb_baseline = XGBRegressor(random_state=42, n_jobs=-1, verbosity=0)baseline_scores = cross_val_score(xgb_baseline, X, y, cv=5, scoring='neg_mean_absolute_error')baseline_mae = -baseline_scores.mean()baseline_std = baseline_scores.std()print(f"Baseline XGBRegressor (5-fold CV)")print(f"Mean Absolute Error: ${baseline_mae:,.2f}")print(f"Standard Deviation: ${baseline_std:,.2f}")print(f"\nDefault hyperparameters:")print(f"  - learning_rate: {xgb_baseline.learning_rate}")print(f"  - n_estimators: {xgb_baseline.n_estimators}")print(f"  - max_depth: {xgb_baseline.max_depth}")

## 5. Learning Rate Search

In [ ]:
# Test different learning rates with 5-fold CVlearning_rates = [0.01, 0.05, 0.1, 0.2, 0.3]lr_results = []for lr in learning_rates:    xgb = XGBRegressor(learning_rate=lr, random_state=42, n_jobs=-1, verbosity=0)    scores = cross_val_score(xgb, X, y, cv=5, scoring='neg_mean_absolute_error')    mae = -scores.mean()    lr_results.append({'learning_rate': lr, 'MAE': mae})    print(f"Learning Rate {lr}: ${mae:,.2f}")# Create results DataFrame and plotlr_df = pd.DataFrame(lr_results)print(f"\n{lr_df.to_string(index=False)}")plt.figure(figsize=(10, 5))plt.plot(lr_df['learning_rate'], lr_df['MAE'], marker='o', linewidth=2, markersize=8)plt.xlabel('Learning Rate', fontsize=12)plt.ylabel('Mean Absolute Error ($)', fontsize=12)plt.title('Learning Rate vs MAE', fontsize=14, fontweight='bold')plt.grid(True, alpha=0.3)plt.tight_layout()plt.show()

## 6. N_Estimators Search

In [ ]:
# Test different numbers of estimators with 5-fold CVn_estimators_list = [25, 50, 75, 100, 150, 200]ne_results = []for ne in n_estimators_list:    xgb = XGBRegressor(n_estimators=ne, random_state=42, n_jobs=-1, verbosity=0)    scores = cross_val_score(xgb, X, y, cv=5, scoring='neg_mean_absolute_error')    mae = -scores.mean()    ne_results.append({'n_estimators': ne, 'MAE': mae})    print(f"N_Estimators {ne}: ${mae:,.2f}")# Create results DataFrame and plotne_df = pd.DataFrame(ne_results)print(f"\n{ne_df.to_string(index=False)}")plt.figure(figsize=(10, 5))plt.plot(ne_df['n_estimators'], ne_df['MAE'], marker='s', linewidth=2, markersize=8)plt.xlabel('Number of Estimators', fontsize=12)plt.ylabel('Mean Absolute Error ($)', fontsize=12)plt.title('N_Estimators vs MAE', fontsize=14, fontweight='bold')plt.grid(True, alpha=0.3)plt.tight_layout()plt.show()

## 7. Max Depth Search

In [ ]:
# Test different max depths with 5-fold CVmax_depths = [2, 3, 4, 5, 6, 8]md_results = []for md in max_depths:    xgb = XGBRegressor(max_depth=md, random_state=42, n_jobs=-1, verbosity=0)    scores = cross_val_score(xgb, X, y, cv=5, scoring='neg_mean_absolute_error')    mae = -scores.mean()    md_results.append({'max_depth': md, 'MAE': mae})    print(f"Max Depth {md}: ${mae:,.2f}")# Create results DataFrame and plotmd_df = pd.DataFrame(md_results)print(f"\n{md_df.to_string(index=False)}")plt.figure(figsize=(10, 5))plt.plot(md_df['max_depth'], md_df['MAE'], marker='^', linewidth=2, markersize=8)plt.xlabel('Max Depth', fontsize=12)plt.ylabel('Mean Absolute Error ($)', fontsize=12)plt.title('Max Depth vs MAE', fontsize=14, fontweight='bold')plt.grid(True, alpha=0.3)plt.tight_layout()plt.show()

## 8. 2D Grid Search: Learning Rate × N_Estimators

In [ ]:
# Perform 2D grid search over learning_rate and n_estimatorslearning_rates_grid = [0.01, 0.05, 0.1, 0.2, 0.3]n_estimators_grid = [25, 50, 75, 100, 150, 200]# Create results matrixgrid_results = np.zeros((len(learning_rates_grid), len(n_estimators_grid)))for i, lr in enumerate(learning_rates_grid):    for j, ne in enumerate(n_estimators_grid):        xgb = XGBRegressor(learning_rate=lr, n_estimators=ne,                           random_state=42, n_jobs=-1, verbosity=0)        scores = cross_val_score(xgb, X, y, cv=5, scoring='neg_mean_absolute_error')        mae = -scores.mean()        grid_results[i, j] = mae        print(f"LR={lr}, NE={ne}: ${mae:,.2f}")# Create heatmapgrid_df = pd.DataFrame(grid_results,                        index=learning_rates_grid,                        columns=n_estimators_grid)plt.figure(figsize=(10, 6))sns.heatmap(grid_df, annot=True, fmt='.0f', cmap='RdYlGn_r', cbar_kws={'label': 'MAE ($)'})plt.xlabel('N_Estimators', fontsize=12)plt.ylabel('Learning Rate', fontsize=12)plt.title('2D Hyperparameter Grid Search: Learning Rate × N_Estimators',           fontsize=14, fontweight='bold')plt.tight_layout()plt.show()# Find and report best hyperparametersbest_idx = np.unravel_index(np.argmin(grid_results), grid_results.shape)best_lr = learning_rates_grid[best_idx[0]]best_ne = n_estimators_grid[best_idx[1]]best_mae = grid_results[best_idx]print(f"\nBest Hyperparameters:")print(f"  - Learning Rate: {best_lr}")print(f"  - N_Estimators: {best_ne}")print(f"  - MAE: ${best_mae:,.2f}")print(f"  - Improvement over baseline: ${baseline_mae - best_mae:,.2f} ({100*(baseline_mae - best_mae)/baseline_mae:.1f}%)")

## 9. Feature Importance

In [ ]:
# Train best model on full dataset and extract feature importancesbest_model = XGBRegressor(learning_rate=best_lr, n_estimators=best_ne,                           random_state=42, n_jobs=-1, verbosity=0)best_model.fit(X, y)# Extract feature importancesimportances = best_model.feature_importances_feature_importance_df = pd.DataFrame({    'Feature': feature_cols,    'Importance': importances}).sort_values('Importance', ascending=True)# Horizontal bar chartplt.figure(figsize=(10, 6))plt.barh(feature_importance_df['Feature'], feature_importance_df['Importance'], color='steelblue')plt.xlabel('Importance', fontsize=12)plt.title('Feature Importances (Best Model)', fontsize=14, fontweight='bold')plt.tight_layout()plt.show()print("\nFeature Importances:")print(feature_importance_df.to_string(index=False))

## 10. Predictions vs Actuals

In [ ]:
# Split data into train/test (80/20)X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)# Train best model on training databest_model_split = XGBRegressor(learning_rate=best_lr, n_estimators=best_ne,                                 random_state=42, n_jobs=-1, verbosity=0)best_model_split.fit(X_train, y_train)# Generate predictionsy_pred = best_model_split.predict(X_test)# Calculate test MAEtest_mae = mean_absolute_error(y_test, y_pred)print(f"Test Set MAE: ${test_mae:,.2f}")# Scatter plot of actual vs predictedplt.figure(figsize=(10, 8))plt.scatter(y_test, y_pred, alpha=0.6, s=50, color='steelblue')# Add 45-degree reference linemin_val = min(y_test.min(), y_pred.min())max_val = max(y_test.max(), y_pred.max())plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')plt.xlabel('Actual Price ($)', fontsize=12)plt.ylabel('Predicted Price ($)', fontsize=12)plt.title('Actual vs Predicted Prices (Test Set)', fontsize=14, fontweight='bold')plt.legend(fontsize=11)plt.grid(True, alpha=0.3)plt.tight_layout()plt.show()print(f"\nModel trained on {len(X_train)} samples, tested on {len(X_test)} samples")